# Обработка исторических данных об играх

- Автор: Плахотнюк Арсений Вячеславович
- Дата: 23.01.2026

### Цели и задачи проекта

- Подготовить данные для дальнейшего анализа развития игровой индустрии.

### Описание данных

Данные содержатся в файле .csv и содержат информацию о продажах игр разных жанров и платформ, а также пользовательские и экспертные оценки игр

### Содержимое проекта


Основные шаги:
- Загрузить данные, ознакомиться в ними
- проверить наличие ошибок, пропусков и дубликатов
- отфильтровать данные
- провести категоризацию данных для более удобного последующего анализа

--- 

## 1. Загрузка данных и знакомство с ними

- Загрузите необходимые библиотеки Python и данные датасета `/datasets/new_games.csv`.


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('https://code.s3.yandex.net/datasets/new_games.csv')

- Познакомьтесь с данными: выведите первые строки и результат метода `info()`.


In [ ]:
df.info()

In [ ]:
df.head(5)

In [ ]:
print(df.shape)
df.isna().sum()

## Анализ

- Предоставлен датасет с 11 колонками и 16956 строками. 

- в датасете присутствуют пропуски в колонках Name, Year of Release, Genre, Critic Score, User Score, Rating. Особенно много пропусков в Critic Score - 8714, User Score - 6804, Rating - 6871

- исходные типы данных не везде корректны. Необходимо ввести корректировки: Year of Release -> Int64, EU sales -> float64, JP sales -> float64, User Score -> float64

---

## 2.  Проверка ошибок в данных и их предобработка


### 2.1. Названия, или метки, столбцов датафрейма

In [ ]:
df.columns

In [ ]:
import re

def to_snake_case(s):
    return '_'.join([w.lower() for w in re.split(r'[,; ]+', s)])

In [ ]:
df.columns = [to_snake_case(col_name) for col_name in df.columns]
df.columns

### 2.2. Типы данных

In [ ]:
# посмотрим на типы данных
df.info()

In [ ]:
tmp = df.copy()

Необходимо ввести корректировки: 
- year_of_release -> int64
- eu_sales -> float64, 
- jp_sales -> float64, 
- user_score -> float64

Возможные причины некорректности типов: если в столбце есть разные типы данных (строки и числа), то обычно все записывается как object - самый общий тип

In [ ]:
# 1) year_of_release
df['year_of_release'].unique() # смотрим на возможные значения. 

In [ ]:
# Смотрим на долю пропусков:
df['year_of_release'].isna().sum() / len(df['year_of_release'])  # доля мала < 2%
# значит можно будет удалить без существенного эффекта для всего объема данных 

In [ ]:
df['year_of_release'] = df['year_of_release'].astype(float).astype('Int64')

In [ ]:
# 2) eu_sales
df['eu_sales'].unique()
# Данные имеют нечисловой тип, поскольку есть пропуски, которые помечены строкой unknown

In [ ]:
np.sum(df['eu_sales'] == 'unknown') / len(df['eu_sales']) # меньше 0.04% пропусков, можно просто удалить

In [ ]:
# меняем unknown на nan-ы и кастуем к нужному типу
df['eu_sales'] = df['eu_sales'].replace(to_replace='unknown', value=np.nan)
df['eu_sales'] = df['eu_sales'].astype(float) 

In [ ]:
df['eu_sales'].isna().sum()  / len(df['eu_sales'])  # сделали пропуски явными

In [ ]:
# 3) jp_sales
df['jp_sales'].unique()

In [ ]:
# 3) jp_sales -- аналогичная ситуация: пропуски помечены как unknown (из-за этого тип object)
np.sum(df['jp_sales'] == 'unknown') / len(df['jp_sales'])

In [ ]:
df['jp_sales'].isna().sum()  # nan-ов нет

In [ ]:
# меняем unknown на nan-ы и кастуем к нужному типу
df['jp_sales'] = df['jp_sales'].replace(to_replace='unknown', value=np.nan)
df['jp_sales'] = df['jp_sales'].astype(float)  

In [ ]:
df['jp_sales'].isna().sum()  / len(df['jp_sales'])  # сделали пропуски явными

In [ ]:
# 4) na_sales - нужного типа и без пропусков
df['na_sales'].isna().sum()

In [ ]:
# 5) other_sales - нужного типа и без пропусков
df['other_sales'].isna().sum()

In [ ]:
# 6) user_score
df['user_score'].unique() # есть nan и строка tbd. Из-за наличия строк tbd тип данных изначально не числовой

In [ ]:
np.sum(df['user_score'] == 'tbd') / len(df['user_score'])  # 15 процентов -- не мало 

In [ ]:
df['user_score'].isna().sum() / len(df['user_score'])  # 40 процентов - даже много 

In [ ]:
# меняем unknown на nan-ы и кастуем к нужному типу
df['user_score'] = df['user_score'].replace(to_replace='tbd', value=np.nan)
df['user_score'] = df['user_score'].astype(float)  

In [ ]:
df['user_score'].isna().sum() / len(df['user_score'])  # теперь все пропуски явные и имеют значение nan

In [ ]:
# 7) genre
df['genre'].unique() # тут все c типами нормально, есть пропуски в виде nan

In [ ]:
# 8) genre
df['rating'].unique()  # пропуски в виде nan. Тип object устраивает, этот столбец со строковыми данными.

In [ ]:
df['rating'].isna().sum() / len(df['rating'])  # около 40%,  что довольно много

In [ ]:
df.head(5)

### 2.3. Наличие пропусков в данных

- Посчитайте количество пропусков в каждом столбце в абсолютных и относительных значениях.


In [ ]:
# кол-во пропусков по абсолютной величине
df.isna().sum() 
# У user_score увеличилось количество nan-ов, поскольку они были преобразованы из строк tbd
# У eu_sales, jp_sales добавились пропуски, поскольку преобразовали из  строк unknown 
# в остальном количество пропусков осталось прежним 

In [ ]:
# кол-во пропусков по относительной величине (проценты от общего числа строк)
df.isna().sum() / len(df) * 100

In [ ]:
# Комментарий ревьюера
def show_missing_stats(tmp0):
    """
    Функция для отображения статистики пропущенных значений в DataFrame.
    """
    missing_stats = pd.DataFrame({
        'Кол-во пропусков': tmp0.isnull().sum(),
        'Доля пропусков': tmp0.isnull().mean()
    })
    missing_stats = missing_stats[missing_stats['Кол-во пропусков'] > 0]
    
    if missing_stats.empty:
        return "Пропусков в данных нет"
    
    # Форматируем при выводе через Styler
    return (missing_stats.style.format({'Доля пропусков': '{:.4f}'}).background_gradient(cmap='coolwarm'))
show_missing_stats(df)

### Анализ пропусков в данных

Где есть пропуски:
- в колонках name, year_of_release, genre, critic_score, user_score, rating, eu_sales, jp_sales, user_score есть пропуски в виде nan-ов

Какие особенности у пропусков:
- name, year_of_release, genre, eu_sales, jp_sales -- пропусков менее 2%.
- Большое количество пропусков в critic_score - 52%, user_score - 54%, rating - 40%. По всей видимости, для анализа представлено много непопулярных игр, оценки по которым редко ставят. 

Что можно сделать с пропусками:
- Пропуски по оценкам можно поставить индикатор -1
- пропущенный рейтинг можно поставить флаг 'Не указано'
- малочисленные пропуски удаляем


In [ ]:
df.head()

In [ ]:
# Обработка пропущенных значений
# Удаляем пропуски для колонок name, year_of_release, genre, jp_sales
df.dropna(subset=['name'], inplace=True)
df['name'].isna().sum()  # проверка

In [ ]:
df.dropna(subset=['year_of_release'], inplace=True)
df['year_of_release'].isna().sum() # проверка

In [ ]:
df.dropna(subset=['genre'], inplace=True)
df['genre'].isna().sum() # проверка

In [ ]:
df.dropna(subset=['jp_sales'], inplace=True)
df['jp_sales'].isna().sum() # проверка

In [ ]:
df.dropna(subset=['eu_sales'], inplace=True)
df['eu_sales'].isna().sum() # проверка

- Где пропусков много и это количественные данные - ставим индикатор -1

In [ ]:
df['critic_score'] = df['critic_score'].fillna(-1)

In [ ]:
df['user_score'] = df['user_score'].fillna(-1)

In [ ]:
# Заполняем флагами
df['rating'] = df['rating'].fillna('Не указано')

In [ ]:
df.isna().sum()  # проверка, что более пропусков нет

In [ ]:
print('Размер исходного датасета: ', len(tmp))
print('Размер датасета после обработки пропусков: ', len(df))
print('Исходный датасет сократился на: ', (len(tmp) - len(df)) / len(tmp) * 100, ' %')

- проверка на дубликаты

In [ ]:
df['genre'].unique()  # есть дубли из-за разного регистра 

In [ ]:
df['genre'] = df['genre'].str.lower()
df['genre'].unique() # дубли устранены

In [ ]:
df['name'].nunique()

In [ ]:
df['name'] = df['name'].apply(to_snake_case)

In [ ]:
df['name'].nunique() # убрали 2 неявных дубликата

In [ ]:
df['rating'].nunique()

In [ ]:
df['rating'] = df['rating'].str.upper()  # названия в верхний

In [ ]:
df['rating'].nunique()  # провел нормализацию, дубликатов по итогу не обнаружено 

In [ ]:
dupl_rows_count = df.duplicated().sum()  # явные дубликаты
total_rows_count = len(df)

In [ ]:
# Чистка от явных дубликатов
df.drop_duplicates(subset=None, keep='first', inplace=True)

### Вывод по устранению дубликатов
- найдены неявные дубликаты в genre, устранены с помощью приведения к одному регистру
- найдено 235 явных дубликатов, что составило 1.4% от общего размера датасета, явные дубли удалены

In [ ]:
# Удалено дубликатов:
dupl_prcnt = dupl_rows_count / total_rows_count * 100
print('Абсолютное количество дубликатов', dupl_rows_count)
print(f'Удаленные дубликаты составили {dupl_prcnt} % от общего числа строк в данных')

In [ ]:
print('Было строк в исходном датасете: ', len(tmp))
print('Осталось строк после обработки: ', len(df))
print('Удалено строк в датасете после обработки: ', len(tmp) - len(df))
print('Процент потерь: ', (len(tmp) - len(df)) / len(tmp) * 100)

In [ ]:
# Комментарий ревьюера 3
# Проверим сколько удалено строк датасета
a, b = len(tmp), len(df)
print(" Было строк в исходном датасете", a,
      '\n', "Осталось строк в датасете после обработки", b,
      '\n', "Удалено строк в датасете после обработки", a-b,
      '\n', "Процент потерь", round((a-b)/a*100, 2))

### Общий промежуточный вывод

Обработка пропусков:
- строки с малочисленными пропусками в столбцах name, year_of_release, genre, eu_sales, jp_sales удалены 
- пропуски в critic_score, user_score и rating заменены на флаги.
- процент устраненных пропусков: 1.69 %

Обработка дублей:
- проведена нормализация данных в колонках genre, name, rating: строковые данные теперь в одном регистре 
- после нормализации удалены дубликаты, процент устраненных дублей:  1.4 %

Итог:
- абсолютное значение потерь после обработки:  522
- процент потерь после обработки: 3.08%

---

## 3. Фильтрация данных

Коллеги хотят изучить историю продаж игр в начале XXI века, и их интересует период с 2000 по 2013 год включительно. Отберите данные по этому показателю. Сохраните новый срез данных в отдельном датафрейме, например `df_actual`.

In [ ]:
df_actual = df[(df['year_of_release'] >= 2000) & (df['year_of_release'] <= 2013)].copy()

In [ ]:
df_actual.head()

---

## 4. Категоризация данных
    
Проведите категоризацию данных:
- Разделите все игры по оценкам пользователей и выделите такие категории: высокая оценка (от 8 до 10 включительно), средняя оценка (от 3 до 8, не включая правую границу интервала) и низкая оценка (от 0 до 3, не включая правую границу интервала).

In [ ]:
df_actual.columns

In [ ]:
def categorize_user_score(row):
    if 8 <= row['user_score'] <= 10:
        return 'Высокая оценка'
    elif 3 <= row['user_score'] < 8:
        return 'Средняя оценка'
    elif 0 <= row['user_score'] < 3:
        return 'Низкая оценка'
    else:
        return 'Нет оценки'

In [ ]:
df_actual['user_score_category'] = df_actual.apply(categorize_user_score, axis=1)
df_actual['user_score_category']

- Разделите все игры по оценкам критиков и выделите такие категории: высокая оценка (от 80 до 100 включительно), средняя оценка (от 30 до 80, не включая правую границу интервала) и низкая оценка (от 0 до 30, не включая правую границу интервала).

In [ ]:
def categorize_critic_score(row):
    if 80 <= row['critic_score'] <= 100:
        return 'Высокая оценка'
    elif 30 <= row['critic_score'] < 80:
        return 'Средняя оценка'
    elif 0 <= row['critic_score'] < 30:
        return 'Низкая оценка'
    else:
        return 'Нет оценки'

In [ ]:
df_actual['critic_score_category'] = df_actual.apply(categorize_critic_score, axis=1)
df_actual['critic_score_category']

In [ ]:
# Считаем игры в категориях user_score
def count_games_with_score(score_category: str, category_name: str):
    """Считаем игры в категории"""
    return len(df_actual[df_actual[score_category] == category_name])


In [ ]:
print(f'Игр с низкой оценкой пользователей: {count_games_with_score("user_score_category", "Низкая оценка")}')
print(f'Игр со средней оценкой пользователей: {count_games_with_score("user_score_category", "Средняя оценка")}')
print(f'Игр с высокой оценкой пользователей: {count_games_with_score("user_score_category", "Высокая оценка")}')

In [ ]:
print(f'Игр с низкой оценкой критиков: {count_games_with_score("critic_score_category", "Низкая оценка")}')
print(f'Игр со средней оценкой критиков: {count_games_with_score("critic_score_category", "Средняя оценка")}')
print(f'Игр с высокой оценкой критиков: {count_games_with_score("critic_score_category", "Высокая оценка")}')

- Выделите топ-7 платформ по количеству игр, выпущенных за весь актуальный период.

In [ ]:
top_7_platforms = df['platform'].value_counts().head(7)

top_7_platforms

---

## 5. Итоговый вывод

В конце напишите основной вывод и отразите, какую работу проделали. Не забудьте указать описание среза данных и новых полей, которые добавили в исходный датасет.

## Итог


1) Исходно предоставлен датасет с 11 колонками и 16956 строками. В датасете присутствуют пропуски в колонках Name, Year of Release, Genre, Critic Score, User Score, Rating. Особенно много пропусков в Critic Score - 8714, User Score - 6804, Rating - 6871. Исходные типы данных не везде корректны. Необходимо ввести корректировки: Year of Release -> Int64, EU sales -> float64, JP sales -> float64, User Score -> float64

2) Анализ и предобработка: 
- найдены пропуски в колонках name, year_of_release, genre, critic_score, user_score, rating, eu_sales, jp_sales, user_score. Малочисленные пропуски удалены, многочисленные заменены на флаги. После обработки пропусков датасет сократился на 1.69%.
- обработка дубликатов. проведена нормализация данных в колонках genre, name, rating: строковые данные теперь в одном регистре. После нормализации удалены дубликаты, процент устраненных дублей: 1.4 %
- Итог после обработки пропусков и дублей: потери в датасете составили: 3.08%

3) Фильтрация: 
- были отобраны данные за релевантный период с 2000 по 2013 год включительно

4) Категоризация. 

- В результате категоризации получены результаты по оценкам за актуальный период:

    - Игр с низкой оценкой пользователей: 116
    - Игр со средней оценкой пользователей: 4078
    - Игр с высокой оценкой пользователей: 2282
    - Игр с низкой оценкой критиков: 55
    - Игр со средней оценкой критиков: 5421
    - Игр с высокой оценкой критиков: 1686
    
- Выделен топ-7 платформ по количеству игр, выпущенных за весь актуальный период:
   
    - PS2     
    - DS      
    - PS3     
    - Wii     
    - X360    
    - PSP     
    - PS      
    